# Revision experiments A - controls, ablations, seeds
**IJIES paper 20264768 - response to reviewer comments**

This notebook answers the comments that need new runs:

| Comment | What this notebook produces |
|---|---|
| R2.2 "comparison not genuinely controlled" | `flan-t5-small` vs `t5-small` (no instruction tuning) vs **`t5-small` random-init (no pretraining)** - identical data, budget, tokenizer max-lens, optimizer, schedule and decoder |
| R2.6 "copy gate claim not statistically supported" | copy gate vs no gate x **3 seeds** + paired bootstrap CIs |
| R2.4 "ablations do not isolate contributions" | **one-factor-at-a-time** from a single BASE: cleaning, no-edit pairs, length penalty, threshold, iterative refinement, stage-2 |
| R2.7 "stage-2 may be dataset adaptation" | stage-2 vs **joint/mixed (rehearsal) training**, + threshold-transfer test |
| R1.9 harm audit | every run reports the **5-way** outcome table (perfect / improved / unchanged / **lateral** / **harm-strict**) |

**Everything is matched:** same corpus subset, same splits, same global batch (128), same number of
optimizer steps, same decoder - so a difference between two rows is attributable to the one factor that changed.

---
### How to run
1. Add data: **`dariocioni/c4200m`** (Add Input -> Datasets -> search `c4200m`).
2. Settings -> Accelerator **GPU T4 x2**, Internet **ON**, Persistence off.
3. `Run All`. Expected **~6-7 h**. Results are written after *every* run, so a timeout still leaves usable output.
4. Download `revA_results.zip` at the end and send it back.

`QUICK_TEST = True` runs the whole matrix in ~25 min to verify it works end-to-end.


In [ ]:
# ============================== CONFIG ==============================
import os, json, time, math, random, gc, shutil, subprocess, sys
from pathlib import Path

NB_VERSION  = "revA_2026_08_03"
OUT         = Path("/kaggle/working")
SEED        = 42
QUICK_TEST  = False          # True -> tiny end-to-end smoke test (~25 min)

# --- matched budget (identical for EVERY run) ---
TRAIN_N     = 150_000 if not QUICK_TEST else 4_000    # training pairs per run
VAL_N       = 2_000   if not QUICK_TEST else 400
EVAL_N      = 3_000   if not QUICK_TEST else 400      # fixed eval subset (all runs share it)
PER_DEVICE_BATCH, GRAD_ACCUM = 16, 4                  # x2 GPUs -> global batch 128
EPOCHS      = 1
LEARNING_RATE, LABEL_SMOOTHING = 3e-4, 0.1
SRC_DROPOUT_P = 0.05
MAX_INPUT_LEN = MAX_TARGET_LEN = 128
PREFIX      = "Correct the grammar in this sentence: "

# --- decoding (identical default for every run) ---
GEN_BATCH   = 64
BASE_GEN    = dict(max_length=128, num_beams=5, no_repeat_ngram_size=3, length_penalty=1.0)
REFINE_MAX_EDITS = 3
LP_SWEEP    = [0.7, 1.0, 1.2]
TAU_SWEEP   = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]

# --- corpus cleaning (same rules as the paper) ---
PER_SHARD   = 20_000 if not QUICK_TEST else 1_500
MIN_WORDS, MAX_WORDS = 4, 50
MAX_LEN_DIFF, MIN_BAG_OVERLAP, MAX_NONASCII_RATIO = 10, 0.5, 0.25
NOEDIT_FRAC = 0.15

# --- stage 2 ---
STAGE2_EPOCHS, STAGE2_LR = 3, 1e-4
BEA_URL = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"

# --- global time guard (Kaggle batch limit is 12h) ---
GLOBAL_BUDGET_H = 10.5
T0 = time.time()
def hours_left(): return GLOBAL_BUDGET_H - (time.time() - T0) / 3600
def budget_ok(need_h=0.6):
    ok = hours_left() > need_h
    if not ok: print(f"!! budget guard: {hours_left():.2f}h left, need {need_h}h -> skipping")
    return ok

RESULTS_JSON = OUT / "revA_results.json"
RESULTS = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else {}
def save_results():
    RESULTS_JSON.write_text(json.dumps(RESULTS, indent=1))

print(f"{NB_VERSION} | QUICK_TEST={QUICK_TEST} | train {TRAIN_N:,} | eval {EVAL_N:,}")
print(f"global batch = {PER_DEVICE_BATCH}*{GRAD_ACCUM}*2 = {PER_DEVICE_BATCH*GRAD_ACCUM*2}"
      f" -> ~{math.ceil(TRAIN_N/(PER_DEVICE_BATCH*GRAD_ACCUM*2))} optimizer steps/run")


In [ ]:
# ============================== INSTALL ==============================
!pip install -q transformers accelerate sentencepiece errant pyarrow 2>/dev/null
import torch, transformers, numpy as np, pandas as pd
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs", torch.cuda.device_count())


In [ ]:
# ============================== IMPORTS / SEEDING ==============================
import numpy as np, pandas as pd, torch, re, unicodedata, tarfile, urllib.request
from collections import Counter
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, set_seed)
from torch.utils.data import Dataset

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); set_seed(s)
seed_all(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"


## 1. Copy-gate module
Written to a file first (importable), so `from_pretrained` works on a custom class.

In [ ]:
%%writefile /kaggle/working/t5_copy_gate.py
# T5 with a pointer-generator copy gate (port of Model 3's mechanism).
# This class lives in a real module file (not the notebook) because recent
# transformers versions inspect the defining module's source file during
# from_pretrained; classes defined in a notebook (__main__) have no __file__.
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration
from transformers.modeling_outputs import Seq2SeqLMOutput


class T5WithCopyGate(T5ForConditionalGeneration):
    """FLAN-T5 + pointer-generator copy gate.

    P(w) = p_gen * softmax(lm_logits)[w] + (1 - p_gen) * sum_{i: src_i == w} attn_i
    attn  = head-averaged cross-attention of the last decoder layer
    p_gen = sigmoid(W_g [decoder_state ; attention_context] + b_g)

    The returned `logits` are log-probabilities of the mixture. The mixture sums
    to ~1, so log_softmax over it is (numerically) the identity: HF generate(),
    beam search, and the Trainer's label smoothing work unchanged.
    """

    def __init__(self, config):
        super().__init__(config)
        self.copy_gate = nn.Linear(2 * config.d_model, 1)
        nn.init.zeros_(self.copy_gate.weight)
        nn.init.constant_(self.copy_gate.bias, 2.0)
        self._last_pgen_mean = None

    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None,
                labels=None, copy_src_ids=None, **kwargs):
        if labels is not None and decoder_input_ids is None and kwargs.get("decoder_inputs_embeds") is None:
            decoder_input_ids = self._shift_right(labels)
        kwargs["output_attentions"] = True
        kwargs["output_hidden_states"] = True
        kwargs["return_dict"] = True
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask,
                                  decoder_input_ids=decoder_input_ids, labels=None, **kwargs)

        if copy_src_ids is None:
            copy_src_ids = input_ids

        if copy_src_ids is None:
            # No source ids available (plain LM usage): keep the vanilla distribution.
            new_logits = outputs.logits
        else:
            attn = outputs.cross_attentions[-1].mean(dim=1).float()        # (B, T, S)
            if attention_mask is not None:
                attn = attn * attention_mask.unsqueeze(1).float()          # zero padded src
            enc_h = outputs.encoder_last_hidden_state.float()              # (B, S, D)
            dec_h = outputs.decoder_hidden_states[-1].float()              # (B, T, D)
            context = torch.bmm(attn, enc_h)                               # (B, T, D)
            p_gen = torch.sigmoid(self.copy_gate(torch.cat([dec_h, context], dim=-1)).float())
            self._last_pgen_mean = float(p_gen.mean().detach())

            probs = torch.softmax(outputs.logits.float(), dim=-1)          # (B, T, V)
            probs = probs * p_gen
            copy_w = attn * (1.0 - p_gen)                                  # (B, T, S)
            idx = copy_src_ids.unsqueeze(1).expand(-1, probs.size(1), -1)  # (B, T, S)
            probs.scatter_add_(2, idx, copy_w)
            new_logits = torch.log(probs + 1e-9)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(new_logits.view(-1, new_logits.size(-1)),
                                   labels.view(-1).to(new_logits.device), ignore_index=-100)

        return Seq2SeqLMOutput(
            loss=loss,
            logits=new_logits,
            past_key_values=outputs.past_key_values,
            decoder_hidden_states=outputs.decoder_hidden_states,
            decoder_attentions=outputs.decoder_attentions,
            cross_attentions=outputs.cross_attentions,
            encoder_last_hidden_state=outputs.encoder_last_hidden_state,
            encoder_hidden_states=outputs.encoder_hidden_states,
            encoder_attentions=outputs.encoder_attentions,
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        model_inputs = super().prepare_inputs_for_generation(input_ids, **kwargs)
        if kwargs.get("copy_src_ids") is not None:
            model_inputs["copy_src_ids"] = kwargs["copy_src_ids"]
        return model_inputs

In [ ]:
sys.path.insert(0, "/kaggle/working")
from t5_copy_gate import T5WithCopyGate
print("T5WithCopyGate imported")


## 2. Data - one fixed corpus, two variants
`clean` = the paper's cleaning filters. `raw` = same pairs, **cleaning disabled** (isolates the
cleaning factor for R2.4). Splits are cut **once** and reused by every run.

In [ ]:
# ============================== DATA ==============================
C4_DIR = Path("/kaggle/input/datasets/dariocioni/c4200m")
files = sorted(C4_DIR.glob("C4_200M.tsv-*")) or sorted(Path("/kaggle/input").rglob("C4_200M.tsv-*"))
assert files, "Attach the c4200m dataset (Add Input -> Datasets -> c4200m)"
print(f"{len(files)} shards")

MOJIBAKE = ("Ã", "â€", "Â", "ï¿½")
RE_MARKUP  = re.compile(r"<[^>]+>|&[a-z]{2,6};|https?://|\{\{|\}\}")
RE_PUNCTRUN= re.compile(r"[!?.,;:]{3,}")
RE_REPEAT  = re.compile(r"\b(\w+)\s+\1\b", re.I)
def nonascii_ratio(s): return sum(ord(c) > 127 for c in s) / max(len(s), 1)
def norm_ws(s): return re.sub(r"\s+", " ", str(s)).strip()

def clean_reason(src, tgt):
    """None = keep. Otherwise the name of the filter that rejected the pair."""
    ls, lt = len(src.split()), len(tgt.split())
    if not (MIN_WORDS <= ls <= MAX_WORDS and MIN_WORDS <= lt <= MAX_WORDS): return "length"
    if src == tgt: return "identical"
    if abs(ls - lt) > MAX_LEN_DIFF: return "len_diff"
    a, b = Counter(src.lower().split()), Counter(tgt.lower().split())
    if sum((a & b).values()) / max(min(ls, lt), 1) < MIN_BAG_OVERLAP: return "low_overlap"
    if RE_PUNCTRUN.search(tgt): return "punct_run_tgt"
    if RE_MARKUP.search(src):   return "markup_src"
    if RE_MARKUP.search(tgt):   return "markup_tgt"
    if RE_REPEAT.search(tgt):   return "repeat_word_tgt"
    if any(m in src for m in MOJIBAKE): return "mojibake_src"
    if any(m in tgt for m in MOJIBAKE): return "mojibake_tgt"
    if nonascii_ratio(tgt) > MAX_NONASCII_RATIO: return "nonascii_tgt"
    return None

def basic_ok(src, tgt):
    """'raw' variant: only the minimal length / non-identical filter (no cleaning)."""
    ls, lt = len(src.split()), len(tgt.split())
    return MIN_WORDS <= ls <= MAX_WORDS and MIN_WORDS <= lt <= MAX_WORDS and src != tgt

need = TRAIN_N + VAL_N + EVAL_N
clean_rows, raw_rows, seen, stats = [], [], set(), Counter()
for f in files:
    kept_c = 0
    for chunk in pd.read_csv(f, sep="\t", header=None, names=["src", "tgt"],
                             dtype=str, quoting=3, on_bad_lines="skip", chunksize=200_000):
        for s, t in zip(chunk["src"], chunk["tgt"]):
            if not isinstance(s, str) or not isinstance(t, str): continue
            s, t = norm_ws(s), norm_ws(t)
            stats["scanned"] += 1
            if basic_ok(s, t) and len(raw_rows) < need: raw_rows.append((s, t))
            r = clean_reason(s, t)
            if r: stats[f"drop_{r}"] += 1; continue
            if s.lower() in seen: stats["drop_dup"] += 1; continue
            seen.add(s.lower()); clean_rows.append((s, t)); kept_c += 1
            if kept_c >= PER_SHARD: break
        if kept_c >= PER_SHARD or len(clean_rows) >= need: break
    print(f"  {f.name}: clean={len(clean_rows):,} raw={len(raw_rows):,}")
    if len(clean_rows) >= need and len(raw_rows) >= need: break

rng = random.Random(SEED); rng.shuffle(clean_rows); rng.shuffle(raw_rows)
clean_rows, raw_rows = clean_rows[:need], raw_rows[:need]
stats["kept_clean"] = len(clean_rows)
def cut(rows):
    return (rows[:TRAIN_N], rows[TRAIN_N:TRAIN_N+VAL_N], rows[TRAIN_N+VAL_N:TRAIN_N+VAL_N+EVAL_N])
TR_C, VA_C, TE_C = cut(clean_rows)
TR_R, _,   _     = cut(raw_rows)          # only the TRAIN half of 'raw' is used
print(f"\nclean: train {len(TR_C):,} val {len(VA_C):,} eval {len(TE_C):,} | raw train {len(TR_R):,}")
json.dump(dict(stats), open(OUT / "revA_filter_stats.json", "w"), indent=1)
print({k: v for k, v in stats.most_common(12)})


In [ ]:
# ---- no-edit (identity) augmentation, used only by the +noedit config ----
def add_noedit(pairs, frac, seed=SEED):
    r = random.Random(seed)
    extra = [(t, t) for _, t in r.sample(pairs, int(len(pairs) * frac))]
    out = pairs + extra; r.shuffle(out); return out

class PairDS(Dataset):
    def __init__(self, pairs, tok, src_dropout_p=0.0, seed=SEED):
        self.p, self.tok, self.dp = pairs, tok, src_dropout_p
        self.rng = random.Random(seed)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        s, t = self.p[i]
        if self.dp > 0 and s != t:                    # dropout only on real edit pairs
            w = [x for x in s.split() if self.rng.random() > self.dp]
            s = " ".join(w) if w else s
        x = self.tok(PREFIX + s, max_length=MAX_INPUT_LEN, truncation=True)
        y = self.tok(text_target=t, max_length=MAX_TARGET_LEN, truncation=True)
        x["labels"] = y["input_ids"]; return x
print("dataset helpers ready")


## 3. Metrics - 5-way outcome table (answers R1.9)
The submitted paper used `harm = else`, which silently bundled **equidistant** predictions
(`d_pred == d_src`) with genuinely worse ones. Here the two are reported separately:

* `improved` : `d_pred <  d_src`
* `lateral`  : `d_pred == d_src` (edited, no change in distance - **not** harmful)
* `harm`     : `d_pred >  d_src` (strictly worse)

In [ ]:
# ============================== METRICS ==============================
def tok_edit(a, b):
    a, b = a.split(), b.split()
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def ngrams(x, n): return Counter(tuple(x[i:i+n]) for i in range(len(x) - n + 1))
def gleu(src, hyp, ref, N=4):
    """Napoles et al. GLEU (single reference)."""
    num = den = 0.0
    for s, h, r in zip(src, hyp, ref):
        s, h, r = s.split(), h.split(), r.split()
        for n in range(1, N + 1):
            hn, rn, sn = ngrams(h, n), ngrams(r, n), ngrams(s, n)
            overlap = sum((hn & rn).values())
            penalty = sum((hn & sn).values()) - sum((hn & sn & rn).values())
            num += max(overlap - penalty, 0); den += max(sum(hn.values()), 1)
    bl = min(1.0, math.exp(1 - sum(len(r.split()) for r in ref) /
                           max(sum(len(h.split()) for h in hyp), 1)))
    return bl * (num / max(den, 1))

def outcome_table(srcs, tgts, preds):
    c = Counter(); n = len(srcs)
    for s, t, p in zip(srcs, tgts, preds):
        ds, dp = tok_edit(s, t), tok_edit(p, t)
        if p.strip() == t.strip():   c["perfect"] += 1
        elif p.strip() == s.strip(): c["unchanged"] += 1
        elif dp < ds:                c["improved"] += 1
        elif dp == ds:               c["lateral"] += 1
        else:                        c["harm"] += 1
    out = {k: 100 * c[k] / n for k in ("perfect", "improved", "unchanged", "lateral", "harm")}
    out["harm_bundled_old"] = out["harm"] + out["lateral"]   # the paper's original definition
    out["helpful"] = out["perfect"] + out["improved"]
    return out
print("metrics ready")


In [ ]:
# ---- ERRANT edit-level P/R/F0.5 ----
import errant, spacy
_ANN = {"a": None}
def ann():
    if _ANN["a"] is None:
        try: nlp = spacy.load("en_core_web_sm")
        except Exception:
            subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
            nlp = spacy.load("en_core_web_sm")
        try:    _ANN["a"] = errant.load("en", nlp)
        except TypeError: _ANN["a"] = errant.load("en")
    return _ANN["a"]

def errant_prf(srcs, hyps, refs):
    a = ann(); tp = fp = fn = 0
    for s, h, r in zip(srcs, hyps, refs):
        so, ho, ro = a.parse(s), a.parse(h), a.parse(r)
        hyp = {(e.o_start, e.o_end, e.c_str) for e in a.annotate(so, ho) if e.type != "noop"}
        gold = {(e.o_start, e.o_end, e.c_str) for e in a.annotate(so, ro) if e.type != "noop"}
        tp += len(hyp & gold); fp += len(hyp - gold); fn += len(gold - hyp)
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    F = (1.25 * P * R / max(0.25 * P + R, 1e-9)) if (P + R) else 0.0
    return dict(P=P, R=R, F05=F, tp=tp, fp=fp, fn=fn)
print("errant ready")


## 4. Model factory - the matched architectural controls (R2.2)

In [ ]:
# ============================== MODELS ==============================
BACKBONES = {
    "flan_t5_small":  dict(name="google/flan-t5-small", pretrained=True),   # proposed
    "t5_small":       dict(name="google-t5/t5-small",   pretrained=True),   # no instruction tuning
    "t5_small_scratch": dict(name="google-t5/t5-small", pretrained=False),  # NO pretraining
}
def build(backbone, use_gate=False, seed=SEED):
    spec = BACKBONES[backbone]; seed_all(seed)
    tok = AutoTokenizer.from_pretrained(spec["name"])
    if use_gate:
        m = T5WithCopyGate.from_pretrained(spec["name"])
    elif spec["pretrained"]:
        m = AutoModelForSeq2SeqLM.from_pretrained(spec["name"])
    else:
        cfg = AutoConfig.from_pretrained(spec["name"])
        m = AutoModelForSeq2SeqLM.from_config(cfg)      # random init, identical architecture
    return tok, m

def n_params(m): return sum(p.numel() for p in m.parameters())
print({k: v["name"] for k, v in BACKBONES.items()})


## 5. Decoding - length penalty / confidence threshold / iterative refinement as switches

In [ ]:
# ============================== DECODE ==============================
@torch.no_grad()
def generate(model, tok, texts, gen_kwargs=None, bs=GEN_BATCH, desc=""):
    model.eval(); gk = dict(BASE_GEN); gk.update(gen_kwargs or {})
    out = []
    for i in range(0, len(texts), bs):
        b = [PREFIX + t for t in texts[i:i+bs]]
        enc = tok(b, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_INPUT_LEN).to(model.device)
        g = model.generate(**enc, **gk)
        out += tok.batch_decode(g, skip_special_tokens=True)
        if desc and i % (bs * 20) == 0: print(f"    {desc} {i}/{len(texts)}", flush=True)
    return [norm_ws(x) for x in out]

@torch.no_grad()
def mean_nll(model, tok, ins, outs, bs=GEN_BATCH):
    """mean per-token NLL of `outs` given `ins` (used for the confidence margin)."""
    model.eval(); res = []
    for i in range(0, len(ins), bs):
        x = tok([PREFIX + t for t in ins[i:i+bs]], return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
        y = tok(text_target=list(outs[i:i+bs]), return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_TARGET_LEN).to(model.device)
        lab = y.input_ids.clone(); lab[lab == tok.pad_token_id] = -100
        logits = model(**x, labels=lab).logits
        lp = torch.log_softmax(logits.float(), -1)
        mask = lab != -100
        tokl = lp.gather(-1, lab.clamp(min=0).unsqueeze(-1)).squeeze(-1) * mask
        res += (tokl.sum(1) / mask.sum(1).clamp(min=1)).tolist()
    return res

def apply_threshold(model, tok, srcs, preds, tau):
    """Revert an edit when logP(pred) - logP(src) < tau  (Eq. 7/8 of the paper)."""
    if tau is None: return preds, 0
    idx = [i for i, (s, p) in enumerate(zip(srcs, preds)) if s.strip() != p.strip()]
    if not idx: return preds, 0
    sub_s = [srcs[i] for i in idx]; sub_p = [preds[i] for i in idx]
    mp = mean_nll(model, tok, sub_s, sub_p); ms = mean_nll(model, tok, sub_s, sub_s)
    out, rev = list(preds), 0
    for k, i in enumerate(idx):
        if (mp[k] - ms[k]) < tau: out[i] = srcs[i]; rev += 1
    return out, rev

def refine(model, tok, srcs, preds, gen_kwargs=None, max_edits=REFINE_MAX_EDITS):
    """Guarded iterative refinement: re-run on own output, accept if <= max_edits tokens change."""
    second = generate(model, tok, preds, gen_kwargs)
    out = [b if tok_edit(a, b) <= max_edits else a for a, b in zip(preds, second)]
    return out
print("decode ready")


## 6. One experiment = train + evaluate under the matched budget

In [ ]:
# ============================== RUNNER ==============================
def evaluate(model, tok, pairs, gen_kwargs=None, tau=None, use_refine=False, desc=""):
    srcs = [s for s, _ in pairs]; tgts = [t for _, t in pairs]
    preds = generate(model, tok, srcs, gen_kwargs, desc=desc)
    if use_refine: preds = refine(model, tok, srcs, preds, gen_kwargs)
    rev = 0
    if tau is not None: preds, rev = apply_threshold(model, tok, srcs, preds, tau)
    r = outcome_table(srcs, tgts, preds)
    r.update(errant_prf(srcs, preds, tgts))
    r["gleu"] = gleu(srcs, preds, tgts)
    r["gleu_src_baseline"] = gleu(srcs, srcs, tgts)
    r["n_reverted"] = rev
    return r, preds

def train_one(tag, backbone="flan_t5_small", use_gate=False, seed=SEED,
              train_pairs=None, val_pairs=None, init_from=None,
              epochs=EPOCHS, lr=LEARNING_RATE, save_dir=None):
    train_pairs = train_pairs if train_pairs is not None else TR_C
    val_pairs   = val_pairs   if val_pairs   is not None else VA_C
    if init_from is not None:                     # stage-2 / continued training
        tok, model = init_from; seed_all(seed)
    else:
        tok, model = build(backbone, use_gate, seed)
    model.to(DEV)
    targs = dict(
        output_dir=str(OUT / f"ckpt_{tag}"), per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=epochs, learning_rate=lr,
        label_smoothing_factor=LABEL_SMOOTHING, lr_scheduler_type="cosine", warmup_ratio=0.03,
        weight_decay=0.01, logging_steps=100, save_strategy="no",
        fp16=torch.cuda.is_available(), report_to=[], seed=seed, dataloader_num_workers=2,
        predict_with_generate=False)
    import inspect
    _p = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    targs["eval_strategy" if "eval_strategy" in _p else "evaluation_strategy"] = "no"
    args = Seq2SeqTrainingArguments(**targs)
    tr = Seq2SeqTrainer(model=model, args=args,
                        train_dataset=PairDS(train_pairs, tok, SRC_DROPOUT_P, seed),
                        data_collator=DataCollatorForSeq2Seq(tok, model=model))
    t0 = time.time(); out = tr.train(); dt = time.time() - t0
    meta = dict(tag=tag, backbone=backbone, use_gate=use_gate, seed=seed,
                n_train=len(train_pairs), steps=int(out.global_step),
                global_batch=PER_DEVICE_BATCH * GRAD_ACCUM * max(torch.cuda.device_count(), 1),
                train_seconds=round(dt, 1), samples_per_second=round(len(train_pairs)*epochs/dt, 2),
                params=n_params(model), train_loss=float(out.training_loss))
    if save_dir:
        model.save_pretrained(save_dir); tok.save_pretrained(save_dir)
    return tok, model, meta

def run(tag, **kw):
    """train + evaluate + record. Skips if already done (safe re-run)."""
    if tag in RESULTS: print(f"== {tag}: already done, skipping"); return
    if not budget_ok(0.7): return
    print(f"\n{'='*70}\n== {tag}   ({hours_left():.2f}h left)\n{'='*70}", flush=True)
    ev = dict(gen_kwargs=kw.pop("gen_kwargs", None), tau=kw.pop("tau", None),
              use_refine=kw.pop("use_refine", False))
    tok, model, meta = train_one(tag, **kw)
    res, preds = evaluate(model, tok, TE_C, desc=tag, **ev)
    RESULTS[tag] = {**meta, **res, **{f"eval_{k}": (v if not isinstance(v, dict) else str(v))
                                      for k, v in ev.items()}}
    save_results()
    pd.DataFrame({"src": [s for s, _ in TE_C], "tgt": [t for _, t in TE_C], "pred": preds}) \
      .to_csv(OUT / f"revA_preds_{tag}.csv", index=False)
    print(f"-> F0.5={res['F05']:.4f} P={res['P']:.3f} R={res['R']:.3f} "
          f"GLEU={res['gleu']:.4f} harm={res['harm']:.2f}% lateral={res['lateral']:.2f}%")
    del model; gc.collect(); torch.cuda.empty_cache()
    return tok
print("runner ready")


## 7. Experiment matrix
### 7a. Architectural controls + copy-gate seeds  (R2.2, R2.6)

In [ ]:
# --- copy gate vs no gate, 3 seeds each (R2.6) ---
for s in [42, 1337, 2024]:
    run(f"flan_nogate_s{s}", backbone="flan_t5_small", use_gate=False, seed=s)
    run(f"flan_gate_s{s}",   backbone="flan_t5_small", use_gate=True,  seed=s)

# --- pretraining controls, identical architecture & budget (R2.2) ---
for s in [42, 1337]:
    run(f"t5_pretrained_s{s}", backbone="t5_small",         seed=s)
    run(f"t5_scratch_s{s}",    backbone="t5_small_scratch", seed=s)

pd.DataFrame(RESULTS).T[["backbone","use_gate","seed","params","steps","F05","P","R","gleu","harm","lateral"]]


### 7b. One-factor-at-a-time ablations (R2.4)
Each row changes **exactly one** thing vs `flan_nogate_s42`.

In [ ]:
# --- training-side factors ---
run("abl_minus_cleaning", backbone="flan_t5_small", train_pairs=TR_R)          # cleaning OFF
run("abl_plus_noedit",    backbone="flan_t5_small", train_pairs=add_noedit(TR_C, NOEDIT_FRAC))

# --- decode-side factors: reuse the BASE checkpoint, no retraining ---
BASE_DIR = OUT / "base_model"
if not BASE_DIR.exists() and budget_ok(0.6):
    tok_b, model_b, meta_b = train_one("base_for_decode", backbone="flan_t5_small",
                                       seed=SEED, save_dir=BASE_DIR)
    del model_b; gc.collect(); torch.cuda.empty_cache()

def decode_variant(tag, gen_kwargs=None, tau=None, use_refine=False):
    if tag in RESULTS: print(f"== {tag}: done"); return
    if not budget_ok(0.25): return
    print(f"\n== {tag} (decode-only, {hours_left():.2f}h left)", flush=True)
    tok = AutoTokenizer.from_pretrained(str(BASE_DIR))
    model = AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR)).to(DEV)
    res, preds = evaluate(model, tok, TE_C, gen_kwargs, tau, use_refine, desc=tag)
    RESULTS[tag] = dict(tag=tag, backbone="flan_t5_small", decode_only=True,
                        gen_kwargs=str(gen_kwargs), tau=tau, use_refine=use_refine, **res)
    save_results()
    print(f"-> F0.5={res['F05']:.4f} P={res['P']:.3f} R={res['R']:.3f} harm={res['harm']:.2f}%")
    del model; gc.collect(); torch.cuda.empty_cache()

decode_variant("abl_decode_base")                                     # LP=1.0, no tau, no refine
for lp in LP_SWEEP:
    decode_variant(f"abl_decode_lp{lp}", gen_kwargs=dict(length_penalty=lp))
decode_variant("abl_decode_refine", use_refine=True)
for tau in [0.05, 0.1, 0.2]:
    decode_variant(f"abl_decode_tau{tau}", tau=tau)


### 7c. Stage-2 vs joint/mixed training + threshold transfer (R2.7)
`stage2` = synthetic then W&I (the paper's recipe). `joint` = both mixed in **one** stage, same
total examples - this is the rehearsal baseline the reviewer asked for.

In [ ]:
# --- W&I+LOCNESS ---
def load_wi():
    d = OUT / "bea19"; d.mkdir(exist_ok=True)
    tar = d / "wi.tar.gz"
    if not (d / "wi+locness").exists():
        urllib.request.urlretrieve(BEA_URL, tar)
        with tarfile.open(tar) as t: t.extractall(d)
    pairs = []
    for m2 in sorted((d / "wi+locness" / "m2").glob("*.train.gold.bea19.m2")):
        src = None; edits = []
        for line in m2.read_text(encoding="utf-8").splitlines():
            if line.startswith("S "):
                if src is not None: pairs.append((src, edits)); edits = []
                src = line[2:].strip()
            elif line.startswith("A "):
                p = line[2:].split("|||")
                if p[1] not in ("noop", "UNK"):
                    a, b = p[0].split(); edits.append((int(a), int(b), p[2]))
        if src is not None: pairs.append((src, edits))
    out = []
    for src, edits in pairs:
        w = src.split()
        for a, b, rep in sorted(edits, key=lambda e: -e[0]):
            w[a:b] = rep.split() if rep else []
        tgt = " ".join(w)
        if src and tgt: out.append((norm_ws(src), norm_ws(tgt)))
    return out

WI = load_wi(); rng = random.Random(SEED); rng.shuffle(WI)
WI_HOLD, WI_TRAIN = WI[:int(len(WI)*0.05)], WI[int(len(WI)*0.05):]
print(f"W&I train {len(WI_TRAIN):,} | holdout {len(WI_HOLD):,}")


In [ ]:
# --- stage 2: continue the BASE checkpoint on W&I ---
if "stage2_wi" not in RESULTS and BASE_DIR.exists() and budget_ok(0.8):
    tok = AutoTokenizer.from_pretrained(str(BASE_DIR))
    model = AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR))
    tok2, model2, meta = train_one("stage2_wi", init_from=(tok, model),
                                   train_pairs=WI_TRAIN, val_pairs=WI_HOLD,
                                   epochs=STAGE2_EPOCHS, lr=STAGE2_LR,
                                   save_dir=OUT / "stage2_model")
    res, preds = evaluate(model2, tok2, TE_C, desc="stage2_wi")     # in-domain (forgetting)
    RESULTS["stage2_wi"] = {**meta, **res}; save_results()
    print(f"-> in-domain F0.5={res['F05']:.4f} (forgetting check)")
    del model2; gc.collect(); torch.cuda.empty_cache()

# --- joint / rehearsal: one stage, synthetic + W&I mixed (same total examples) ---
if "joint_mixed" not in RESULTS and budget_ok(0.8):
    k = min(len(WI_TRAIN), TRAIN_N // 4)
    mixed = TR_C[:TRAIN_N - k] + WI_TRAIN[:k]; random.Random(SEED).shuffle(mixed)
    run("joint_mixed", backbone="flan_t5_small", train_pairs=mixed)


In [ ]:
# --- threshold transfer: tune tau on the synthetic val, apply UNCHANGED to W&I holdout (R2.7) ---
if "tau_transfer" not in RESULTS and BASE_DIR.exists() and budget_ok(0.4):
    tok = AutoTokenizer.from_pretrained(str(BASE_DIR))
    model = AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR)).to(DEV)
    vs = [s for s, _ in VA_C][:800]; vt = [t for _, t in VA_C][:800]
    vp = generate(model, tok, vs, desc="tau-tune")
    best = (None, -1)
    for tau in TAU_SWEEP:
        p2, _ = apply_threshold(model, tok, vs, vp, tau)
        f = errant_prf(vs, p2, vt)["F05"]
        print(f"   tau={tau}: val F0.5={f:.4f}")
        if f > best[1]: best = (tau, f)
    tau_star = best[0]; print(f"  -> tau* = {tau_star} (tuned on synthetic val only)")
    hs = [s for s, _ in WI_HOLD][:800]; ht = [t for _, t in WI_HOLD][:800]
    hp = generate(model, tok, hs, desc="wi-holdout")
    no_t = errant_prf(hs, hp, ht)
    p2, rev = apply_threshold(model, tok, hs, hp, tau_star)
    with_t = errant_prf(hs, p2, ht)
    RESULTS["tau_transfer"] = dict(tag="tau_transfer", tau_star=tau_star,
                                   val_F05=best[1], wi_F05_no_threshold=no_t["F05"],
                                   wi_F05_with_transferred_tau=with_t["F05"],
                                   wi_P_no=no_t["P"], wi_P_with=with_t["P"], n_reverted=rev)
    save_results(); print(RESULTS["tau_transfer"])
    del model; gc.collect(); torch.cuda.empty_cache()


## 8. Significance - paired bootstrap (R2.4, R2.6)
Small differences need uncertainty. This resamples the **same** evaluation sentences for both
systems (paired) and reports the 95% CI of the F0.5 difference and a bootstrap p-value.

In [ ]:
# ============================== BOOTSTRAP ==============================
def paired_bootstrap(tag_a, tag_b, n_boot=1000, seed=SEED):
    fa, fb = OUT / f"revA_preds_{tag_a}.csv", OUT / f"revA_preds_{tag_b}.csv"
    if not (fa.exists() and fb.exists()): return None
    A, B = pd.read_csv(fa), pd.read_csv(fb)
    srcs = A["src"].astype(str).tolist(); tgts = A["tgt"].astype(str).tolist()
    pa = A["pred"].astype(str).tolist();  pb = B["pred"].astype(str).tolist()
    a = ann(); per = []
    for s, t, x, y in zip(srcs, tgts, pa, pb):
        so, to = a.parse(s), a.parse(t)
        gold = {(e.o_start, e.o_end, e.c_str) for e in a.annotate(so, to) if e.type != "noop"}
        row = []
        for p in (x, y):
            hyp = {(e.o_start, e.o_end, e.c_str) for e in a.annotate(so, a.parse(p)) if e.type != "noop"}
            row.append((len(hyp & gold), len(hyp - gold), len(gold - hyp)))
        per.append(row)
    def f05(c):
        tp, fp, fn = c
        P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
        return (1.25 * P * R / max(0.25 * P + R, 1e-9)) if (P + R) else 0.0
    tot = lambda idx, ids: tuple(sum(per[i][idx][k] for i in ids) for k in range(3))
    ids0 = range(len(per)); obs = f05(tot(0, ids0)) - f05(tot(1, ids0))
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(n_boot):
        ids = rng.integers(0, len(per), len(per))
        diffs.append(f05(tot(0, ids)) - f05(tot(1, ids)))
    diffs = np.array(diffs)
    return dict(a=tag_a, b=tag_b, delta_F05=obs,
                ci_low=float(np.percentile(diffs, 2.5)), ci_high=float(np.percentile(diffs, 97.5)),
                p_value=float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))))

BOOT = []
for pair in [("flan_nogate_s42", "flan_gate_s42"),
             ("flan_nogate_s1337", "flan_gate_s1337"),
             ("flan_nogate_s2024", "flan_gate_s2024"),
             ("flan_nogate_s42", "t5_pretrained_s42"),
             ("flan_nogate_s42", "t5_scratch_s42"),
             ("abl_decode_base", "abl_decode_refine"),
             ("flan_nogate_s42", "abl_minus_cleaning"),
             ("flan_nogate_s42", "abl_plus_noedit")]:
    r = paired_bootstrap(*pair)
    if r: BOOT.append(r); print(f"{r['a']:22s} vs {r['b']:22s} dF0.5={r['delta_F05']:+.4f} "
                                f"95%CI[{r['ci_low']:+.4f},{r['ci_high']:+.4f}] p={r['p_value']:.3f}")
json.dump(BOOT, open(OUT / "revA_bootstrap.json", "w"), indent=1)


In [ ]:
# ============================== SEED SUMMARY + EXPORT ==============================
df = pd.DataFrame(RESULTS).T
df.to_csv(OUT / "revA_all_runs.csv")

def seed_summary(prefix):
    rows = {k: v for k, v in RESULTS.items() if k.startswith(prefix)}
    if not rows: return None
    f = np.array([r["F05"] for r in rows.values()], dtype=float)
    return dict(config=prefix, n_seeds=len(f), F05_mean=f.mean(), F05_std=f.std(ddof=1) if len(f) > 1 else 0.0)
SUM = [s for s in (seed_summary(p) for p in
       ["flan_nogate_s", "flan_gate_s", "t5_pretrained_s", "t5_scratch_s"]) if s]
print(pd.DataFrame(SUM).to_string(index=False))
json.dump(SUM, open(OUT / "revA_seed_summary.json", "w"), indent=1)

import zipfile
with zipfile.ZipFile(OUT / "revA_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["revA_*.json", "revA_*.csv"]:
        for p in OUT.glob(pat):
            if p.name != "revA_results.zip": z.write(p, p.name)
print("\nDONE ->", OUT / "revA_results.zip", f"| {hours_left():.2f}h to spare")
print("Send back revA_results.zip (revA_all_runs.csv, revA_bootstrap.json, revA_seed_summary.json).")
